# H3a: Per-SV Absolute Utilization Under Momentum Updates

This notebook is the notebook counterpart to `experiments/Tier_1_Core_Mechanism_Tests/H3a_PER_SV_GRADIENT_UTILIZATION/run_experiment.py`.

## Scope and identity
- It imports the script and uses its returned `run_experiment()` results instead of reimplementing the training loop.
- The preserved toy experiment is a deep linear regression problem with momentum-based updates for three methods: `sgd`, `muon`, and `norm_sgd`.
- The measured quantity is **not update energy**. It is the normalized distribution of absolute singular-direction coefficients
  \[
  p_i \propto |u_i^\top \Delta W v_i|
  \]
  where `u_i, v_i` come from the **current gradient** SVD.
- A separate single-gradient sanity check is used to illustrate the simpler no-momentum theory. The main experiment is stricter and different: it studies momentum-accumulated updates projected onto a changing singular-vector basis.

## Questions asked here
1. Does the no-momentum theory sanity check behave as expected?
2. What does the actual momentum-based toy training experiment produce?
3. Do the legacy thresholds T1/T2/T3 pass under the implemented metric?

In [ ]:
from pathlib import Path
import importlib.util
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np

RELATIVE_SCRIPT = Path(
    'experiments/Tier_1_Core_Mechanism_Tests/H3a_PER_SV_GRADIENT_UTILIZATION/run_experiment.py'
)


def find_repo_root(start_path: Path) -> Path:
    start_path = start_path.resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / RELATIVE_SCRIPT).exists():
            return candidate
    raise FileNotFoundError(
        f'Could not locate repository root containing {RELATIVE_SCRIPT} from {start_path}'
    )


NOTEBOOK_CWD = Path.cwd().resolve()
REPO_ROOT = find_repo_root(NOTEBOOK_CWD)
SCRIPT_PATH = REPO_ROOT / RELATIVE_SCRIPT

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

spec = importlib.util.spec_from_file_location('h3a_run_experiment_module', SCRIPT_PATH)
h3a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h3a)

print('Notebook-safe import complete.')
print(f'  cwd:          {NOTEBOOK_CWD}')
print(f'  repo_root:    {REPO_ROOT}')
print(f'  script path:  {SCRIPT_PATH}')
print(f'  python:       {platform.python_version()}')
print(f'  numpy:        {np.__version__}')
print(f'  matplotlib:   {plt.matplotlib.__version__}')

# Reproducibility, runtime, and configuration

The experiment below uses the script's default configuration and deterministic seed list. The script entrypoint remains:

```bash
python experiments/Tier_1_Core_Mechanism_Tests/H3a_PER_SV_GRADIENT_UTILIZATION/run_experiment.py
```

The main workload is CPU-only NumPy linear algebra. The dominant cost is repeated SVDs for every layer, step, seed, and method.

In [ ]:
config = h3a.get_default_config()
seeds = h3a.get_default_seeds(config['num_seeds'])

total_layer_step_measurements_per_method = (
    config['num_steps'] * config['num_layers'] * config['num_seeds']
)
total_svd_calls_across_methods = total_layer_step_measurements_per_method * len(config['methods'])

print('Default configuration')
print('-' * 80)
print(f"counterpart script: {RELATIVE_SCRIPT}")
print(f"seeds:              {seeds}")
print(f"dimension:          {config['dim']}")
print(f"layers:             {config['num_layers']}")
print(f"steps:              {config['num_steps']}")
print(f"batch size:         {config['batch_size']}")
print(f"learning rate:      {config['lr']}")
print(f"momentum:           {config['momentum']}")
print(f"Newton-Schulz iters:{config['ns_iters']}")
print(f"methods:            {config['methods']}")
print(f"H_max:              {config['h_max_bits']:.3f} bits")
print()
print('Workload accounting')
print('-' * 80)
print(
    'per-method layer/step measurements: '
    f"{config['num_steps']} x {config['num_layers']} x {config['num_seeds']} = "
    f"{total_layer_step_measurements_per_method}"
)
print(f'total main-experiment SVD evaluations across methods: ~{total_svd_calls_across_methods}')

# Theory sanity check: single gradient, no momentum

The original narrative is most accurate in the simpler setting where one fixed gradient is examined directly:

- raw SGD update: `ΔW = G`
- normalized raw gradient: `ΔW = G / ||G||_F`
- orthogonalized gradient: `ΔW ≈ UV^T`

In that setting, raw SGD and normalized raw gradient should induce the **same normalized utilization distribution**, while the orthogonalized update should be much closer to uniform across singular directions.

This is a useful sanity check, but it is **not** the same as the momentum-based training metric used in the main experiment below.

In [ ]:
theory = h3a.run_theory_sanity_check(seed=1234)

print('Single-gradient sanity check')
print('-' * 80)
print(f"seed: {theory['seed']}, dim: {theory['dim']}, ns_iters: {theory['ns_iters']}")
print(
    'raw vs normalized raw-gradient L1 difference in normalized utilization: '
    f"{theory['checks']['raw_vs_normalized_l1_difference']:.3e}"
)
print(
    'orthogonalized vs uniform max absolute deviation: '
    f"{theory['checks']['orthogonalized_vs_uniform_max_abs_deviation']:.3e}"
)
print()
print('Entropy comparison (bits)')
print(
    f"  raw SGD:                {theory['raw_sgd']['entropy']:.3f}\n"
    f"  normalized raw grad:    {theory['normalized_raw_gradient']['entropy']:.3f}\n"
    f"  orthogonalized grad:    {theory['orthogonalized_gradient']['entropy']:.3f}\n"
    f"  uniform maximum:        {theory['checks']['expected_uniform_entropy_bits']:.3f}"
)

dim = theory['dim']
x = np.arange(1, dim + 1)
uniform = np.ones(dim) / dim

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(x, theory['raw_sgd']['distribution'], marker='o', label='raw SGD', linewidth=1.5)
axes[0].plot(
    x,
    theory['normalized_raw_gradient']['distribution'],
    marker='s',
    label='normalized raw gradient',
    linewidth=1.2,
    alpha=0.8,
)
axes[0].set_title('Same normalized utilization after raw scaling')
axes[0].set_xlabel('Singular direction index')
axes[0].set_ylabel('Normalized absolute utilization')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(
    x,
    theory['orthogonalized_gradient']['distribution'],
    marker='o',
    label='orthogonalized gradient',
    linewidth=1.5,
)
axes[1].plot(x, uniform, '--', label='uniform reference', linewidth=1.5)
axes[1].set_title('Orthogonalized update approaches uniform utilization')
axes[1].set_xlabel('Singular direction index')
axes[1].set_ylabel('Normalized absolute utilization')
axes[1].grid(alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

# Main experiment: momentum-based training metric

We now run the actual toy experiment preserved in the script. Here the update is the optimizer's **actual momentum-based training update**, and utilization is measured in the **current-step gradient basis**. Because momentum mixes information across time while the singular-vector basis can drift across steps, the clean single-gradient theory should not be assumed to transfer unchanged.

In [ ]:
results = h3a.run_experiment(include_theory_sanity=False)
config = results['config']
methods = results['methods']
method_labels = results['method_labels']
h_max = config['h_max_bits']
colors = {
    'sgd': '#1f77b4',
    'norm_sgd': '#ff7f0e',
    'muon': '#2ca02c',
}

print('Main experiment complete.')
print('-' * 80)
print(f"runtime:           {results['runtime_seconds']:.2f} s")
print(f"pooled records:    {len(results['pooled_records'])}")
print(f"per-step records:  {len(results['per_step_records'])}")
print(f"per-seed summaries:{len(results['per_seed_summaries'])}")
print(f"snapshots stored:  {len(results['snapshot_records'])}")
print(f"failures logged:   {len(results['failures'])}")
if results['failures']:
    print('First failure record:')
    print(results['failures'][0])
else:
    print('No divergence failures were recorded under the default configuration.')

# Aggregate summaries

The script returns both raw and aggregated structures:

- `pooled_records`: one record per `(method, seed, step, layer)` measurement
- `per_step_records`: one record per `(method, seed, step)` after averaging across layers
- `per_seed_summaries`: one record per `(method, seed)`
- `per_step_summary_by_method`: seed-aggregated trajectories used for the plots below
- `tests`: descriptive T1/T2/T3 verdicts carried over from the original script

In [ ]:
print('Pooled summary by method')
print('-' * 98)
print(
    f"{'method':<12} {'label':<38} {'mean H':>8} {'H/H_max':>8} "
    f"{'mean top-1':>12} {'std H':>8} {'N':>8}"
)
for method in methods:
    row = results['pooled_summary'][method]
    print(
        f"{method:<12} {method_labels[method]:<38} {row['mean_entropy']:>8.3f} "
        f"{row['mean_entropy_over_hmax']:>8.3f} "
        f"{row['mean_top1_abs_utilization_fraction']:>12.4f} "
        f"{row['std_entropy']:>8.3f} {row['num_measurements']:>8d}"
    )

entropy_order = sorted(
    methods,
    key=lambda name: results['pooled_summary'][name]['mean_entropy'],
    reverse=True,
)
print()
print('Observed pooled entropy ordering: ' + ' > '.join(entropy_order))
print(f'H_max = {h_max:.3f} bits')

print('\nPer-seed summaries')
print('-' * 98)
print(
    f"{'seed':<8} {'method':<12} {'mean H':>8} {'H/H_max':>8} "
    f"{'mean top-1':>12} {'std H':>8} {'completed steps':>16}"
)
for method in methods:
    for row in results['per_seed_summary_by_method'][method]:
        print(
            f"{row['seed']:<8d} {method:<12} {row['mean_entropy']:>8.3f} "
            f"{row['mean_entropy_over_hmax']:>8.3f} "
            f"{row['mean_top1_abs_utilization_fraction']:>12.4f} "
            f"{row['std_entropy']:>8.3f} {row['completed_steps']:>16d}"
        )

# Per-step entropy trajectory

This plot uses the script's `per_step_summary_by_method`, which averages per-step entropy across seeds after each seed already averaged across layers. The shaded band shows ±1 standard deviation across seeds.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for method in methods:
    rows = results['per_step_summary_by_method'][method]
    steps = np.array([row['step'] for row in rows])
    mean_entropy = np.array([row['mean_entropy'] for row in rows])
    std_entropy = np.array([row['std_entropy'] for row in rows])

    ax.plot(steps, mean_entropy, label=method_labels[method], color=colors[method], linewidth=2)
    ax.fill_between(
        steps,
        mean_entropy - std_entropy,
        mean_entropy + std_entropy,
        color=colors[method],
        alpha=0.18,
    )

ax.axhline(h_max, color='black', linestyle='--', linewidth=1, label='uniform maximum')
ax.set_title('Per-step mean utilization entropy')
ax.set_xlabel('Training step')
ax.set_ylabel('Entropy of normalized absolute utilization (bits)')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Per-step top-1 absolute-utilization trajectory

The quantity below is the mean top singular-direction share of the normalized **absolute utilization** distribution. It is intentionally labeled this way to avoid the earlier overclaim that it was update "energy".

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for method in methods:
    rows = results['per_step_summary_by_method'][method]
    steps = np.array([row['step'] for row in rows])
    mean_top1 = np.array([row['mean_top1_abs_utilization_fraction'] for row in rows])
    std_top1 = np.array([row['std_top1_abs_utilization_fraction'] for row in rows])

    ax.plot(steps, mean_top1, label=method_labels[method], color=colors[method], linewidth=2)
    ax.fill_between(
        steps,
        mean_top1 - std_top1,
        mean_top1 + std_top1,
        color=colors[method],
        alpha=0.18,
    )

ax.axhline(1.0 / config['dim'], color='black', linestyle='--', linewidth=1, label='uniform reference = 1/dim')
ax.set_title('Per-step mean top-1 absolute-utilization fraction')
ax.set_xlabel('Training step')
ax.set_ylabel('Top-1 share of normalized absolute utilization')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

# Seed-level summaries

Because the toy study uses only 5 seeds, seed-level summaries are worth showing explicitly. These are not formal confidence intervals, but they help separate pooled within-run variation from across-seed consistency.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for method in methods:
    rows = results['per_seed_summary_by_method'][method]
    seed_values = [row['seed'] for row in rows]
    mean_entropy = [row['mean_entropy'] for row in rows]
    mean_top1 = [row['mean_top1_abs_utilization_fraction'] for row in rows]

    axes[0].plot(
        seed_values,
        mean_entropy,
        marker='o',
        linewidth=1.8,
        color=colors[method],
        label=method_labels[method],
    )
    axes[1].plot(
        seed_values,
        mean_top1,
        marker='o',
        linewidth=1.8,
        color=colors[method],
        label=method_labels[method],
    )

axes[0].axhline(h_max, color='black', linestyle='--', linewidth=1, label='uniform maximum')
axes[0].set_title('Per-seed mean entropy')
axes[0].set_xlabel('Seed')
axes[0].set_ylabel('Mean entropy (bits)')
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)

axes[1].axhline(1.0 / config['dim'], color='black', linestyle='--', linewidth=1, label='uniform reference = 1/dim')
axes[1].set_title('Per-seed mean top-1 absolute-utilization fraction')
axes[1].set_xlabel('Seed')
axes[1].set_ylabel('Mean top-1 fraction')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# T1/T2/T3 verdicts and conclusion

The original threshold checks are retained as descriptive legacy tests, but the notebook now treats them as empirical verdicts rather than assumptions. The conclusion should track the observed output, not the hoped-for story.

In [ ]:
print('Legacy threshold verdicts (descriptive)')
print('-' * 80)
for key in ['T1', 'T2', 'T3']:
    row = results['tests'][key]
    status = 'PASS' if row['passed'] else 'FAIL'
    print(f'{key}: {status}')
    print(f"  {row['description']}")
    if key == 'T1':
        print(
            f"  actual gap = {row['actual_gap_bits']:.3f} bits; "
            f"required > {row['required_gap_bits']:.3f} bits"
        )
    elif key == 'T2':
        print(
            f"  muon mean entropy = {row['actual_muon_mean_entropy']:.3f}; "
            f"threshold = {row['threshold_entropy_bits']:.3f}; "
            f"fraction of H_max = {row['actual_fraction_of_hmax']:.3f}"
        )
    elif key == 'T3':
        print(
            '  sgd mean top-1 absolute-utilization fraction = '
            f"{row['actual_sgd_mean_top1_abs_utilization_fraction']:.4f}; "
            f"threshold > {row['threshold_top1_abs_utilization_fraction']:.4f}"
        )
    print()

all_pass = all(results['tests'][key]['passed'] for key in ['T1', 'T2', 'T3'])
entropy_order = sorted(
    methods,
    key=lambda name: results['pooled_summary'][name]['mean_entropy'],
    reverse=True,
)
ordering_text = ' > '.join(entropy_order)

print('Conclusion')
print('-' * 80)
if all_pass:
    print('Under the current implementation and defaults, the advertised legacy thresholds all pass.')
else:
    print(
        'Under the current implementation and default toy setup, the advertised legacy expectations '
        'are NOT supported as written.'
    )
print(f'Observed pooled entropy ordering: {ordering_text}')
print(
    'The clean single-gradient sanity check still shows why the original theory was tempting: '
    'raw scaling preserves the normalized utilization distribution, while orthogonalization pushes it '
    'toward uniformity.'
)
print(
    'However, the actual main experiment is different: it measures momentum-based training updates in '
    'the current gradient basis, so basis drift and temporal mixing matter.'
)
print(
    'Therefore the honest first-pass reading is: this notebook preserves the toy experiment, presents it '
    'more carefully, and shows that the current implementation does not by itself validate the stronger '
    'spectral-democracy claims previously implied by the narrative.'
)
print(
    'Finally, the reported top-1 quantity should be read as a top-1 absolute-utilization fraction, not '
    'as squared update energy.'
)